In [ ]:
%pip install scikit-allel

In [ ]:
import pandas as pd
import numpy as np
import allel

In [ ]:
# metadata_path = '../../../results/config/metadata.qcpass.tsv'
# bed_targets_path = "../../../config/ag-vampir.bed"
# vcf_path = "../../../results/vcfs/targets/ag-vampir-002.annot.vcf"
wkdir = "../../.."
# cohort_cols = 'location,taxon'

### Build decision tree model for gambiae, coluzzii, arab and melas

In [ ]:
ampseq_loci = np.load("./agvampir-ampseq-loci.npy")
alts_wgs = np.load("./agvampir-alts-wgs.npy")
gn_wgs = np.load("./agvampir-gn-wgs.npy")

df_samples = pd.read_csv("./agvampir-df-samples.csv", index_col=0)

# metadata = pd.read_csv(metadata_path, sep="\t")

# import json
# with open(f"{wkdir}/config/metadata_colours.json", 'r') as f:
#     color_mapping = json.load(f)

# targets = pd.read_csv(bed_targets_path, sep="\t", header=None)
# targets.columns = ['contig', 'start', 'end', 'amplicon', 'mutation', 'ref', 'alt']

# gn_amp, pos, gn_contigs, metadata, refs, alts_amp, ann = amp.load_vcf(vcf_path=vcf_path, metadata=metadata)
# samples = metadata['sample_id'].to_list()

# # alts = np.concatenate([refs.reshape(refs.shape[0], -1), alts_amp], axis=1)

# wgs_pos  = [int(s.split(":")[1].split("-")[0]) for s in ampseq_loci]
# assert all(np.array(wgs_pos) == pos)

In [ ]:
import numba
@numba.jit(nopython=True)
def _melt_gt_counts(gt_counts):
    n_snps, n_samples, n_alleles = gt_counts.shape
    # Use a float array to allow NaN values
    melted_counts = np.full((n_snps * (n_alleles - 1), n_samples), np.nan, dtype=np.float64)

    for i in range(n_snps):
        for j in range(n_samples):
            for k in range(n_alleles - 1):
                # Check if the genotype count is valid (== 2)
                if gt_counts[i][j].sum() == 2:
                    melted_counts[(i * (n_alleles - 1)) + k][j] = gt_counts[i][j][k + 1]
                else:
                    # Assign NaN for missing or invalid data
                    melted_counts[(i * (n_alleles - 1)) + k][j] = np.nan

    return melted_counts

In [ ]:
taxons = df_samples.taxon.unique()
taxons = [t for t in taxons if t not in ['unassigned', 'gcx4', 'fontenillei', 'quadriannulatus', 'merus']]
mask = df_samples.eval("taxon in @taxons")
df_samples = df_samples[mask]

In [ ]:
df_samples.taxon.value_counts()

In [ ]:
df_samples = df_samples.reset_index()

In [ ]:
import allel
gn_wgs = allel.GenotypeArray(gn_wgs)
gn_wgs = gn_wgs.compress(mask, axis=1)

gn_counts = gn_wgs.to_allele_counts()
gn_counts = _melt_gt_counts(gn_counts.values)

# create labels
alts_flat = alts_wgs[:, 1:].flatten()
labels = np.array([f"{np.repeat(ampseq_loci, 3)[i]}:{alts_flat[i]}" for i in range(len(alts_flat))])

# make dataframe
df_vampir = pd.DataFrame(gn_counts, index=labels, columns=df_samples.sample_id)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectFromModel
import xgboost as xgb
from scipy.stats import uniform, randint



def analyze_taxon_prediction(snp_data, taxon_labels, model_type='decision_tree', test_size=0.2, random_state=42, cv=5):
    """
    Analyze taxon prediction using either Decision Tree or XGBoost classifier.

    Parameters:
    -----------
    snp_data : pandas DataFrame
        DataFrame with SNPs as rows and samples as columns
    taxon_labels : list or array
        List of taxon labels corresponding to each sample
    model_type : str, optional (default='decision_tree')
        Type of model to use. Either 'decision_tree' or 'xgboost'
    test_size : float, optional (default=0.2)
        Proportion of data to use for testing
    random_state : int, optional (default=42)
        Random state for reproducibility
    cv : int, optional (default=5)
        Number of cross-validation folds

    Returns:
    --------
    dict
        Dictionary containing optimized model and evaluation metrics
    """
    # Prepare the data
    X = snp_data.T  # Transpose so samples are rows and SNPs are columns

    # Encode taxon labels
    le = LabelEncoder()
    y = le.fit_transform(taxon_labels)

    # Define model and parameter grid based on model_type
    if model_type.lower() == 'decision_tree':
        base_model = DecisionTreeClassifier(random_state=random_state)
        param_grid = {
            'max_depth': [3, 5, 7, 10, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'criterion': ['gini', 'entropy'],
            'class_weight': ['balanced', None]
        }
    elif model_type.lower() == 'xgboost':
        base_model = xgb.XGBClassifier(random_state=random_state)
        param_grid = {
            'max_depth': [3, 5, 7, 9],
            'learning_rate': [0.01, 0.1, 0.3],
            'n_estimators': [100, 200, 300],
            'min_child_weight': [1, 3, 5],
            'subsample': [0.6, 0.8, 1.0],
            'colsample_bytree': [0.6, 0.8, 1.0],
            'gamma': [0, 0.1, 0.2]
        }
    else:
        raise ValueError("model_type must be either 'decision_tree' or 'xgboost'")

    # Initial model for feature selection
    initial_clf = base_model

    # # Perform feature selection
    selector = SelectFromModel(initial_clf, prefit=False, threshold='mean')
    selector.fit(X, y)
    X_selected = selector.transform(X)
    selected_features = X.columns[selector.get_support()].tolist()

    # Split the data using selected features
    X_train, X_test, y_train, y_test = train_test_split(
        X_selected, y, test_size=test_size, random_state=random_state, stratify=y
    )

    # Grid search with cross-validation
    grid_search = GridSearchCV(
        base_model,
        param_grid=param_grid,
        cv=cv,
        scoring='f1_weighted',
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    # Get best model
    best_clf = grid_search.best_estimator_

    # Make predictions
    y_pred = best_clf.predict(X_test)

    # Cross-validation scores
    cv_scores = cross_val_score(
        best_clf, X_selected, y,
        cv=cv, scoring='f1_weighted'
    )

    # Get feature importance for selected features
    feature_importance = pd.DataFrame({
        'SNP': selected_features,
        'importance': best_clf.feature_importances_
    }).sort_values('importance', ascending=False)

    # Overall evaluation
    overall_report = classification_report(
        y_test,
        y_pred,
        target_names=le.classes_,
        output_dict=True
    )

    # Per-taxon evaluation
    per_taxon_results = {}
    for taxon in le.classes_:
        taxon_idx = le.transform([taxon])[0]
        y_test_binary = (y_test == taxon_idx).astype(int)
        y_pred_binary = (y_pred == taxon_idx).astype(int)

        taxon_report = classification_report(
            y_test_binary,
            y_pred_binary,
            labels=np.unique(y_test),  # Use only the labels present in y_test
            target_names=le.inverse_transform(np.unique(y_test)),
            output_dict=True
        )
        per_taxon_results[taxon] = taxon_report

    # Confusion matrix
    conf_matrix = pd.DataFrame(
        confusion_matrix(y_test, y_pred),
        columns=le.classes_,
        index=le.classes_
    )

    # Model optimization results
    optimization_results = {
        'model_type': model_type,
        'best_params': grid_search.best_params_,
        'cv_scores_mean': cv_scores.mean(),
        'cv_scores_std': cv_scores.std(),
        'selected_features': selected_features,
        'cv_results': pd.DataFrame(grid_search.cv_results_)
    }

    return {
        'model': best_clf,
        'feature_importance': feature_importance,
        'overall_report': overall_report,
        'per_taxon_results': per_taxon_results,
        'confusion_matrix': conf_matrix,
        'sample_counts': pd.Series(taxon_labels).value_counts(),
        'label_encoder': le,
        'optimization_results': optimization_results,
        # 'train_indices': X_train,
        # 'test_indices': X_test,
        # 'X':X_selected,
        # 'y':y
    }

def print_optimization_results(results):
    """Print comprehensive optimization and evaluation results"""
    print("Model Type:", results['optimization_results']['model_type'])
    print("\nModel Optimization Results:")
    print("--------------------------")
    print(f"Best Parameters: {results['optimization_results']['best_params']}")
    print(f"Cross-validation Score: {results['optimization_results']['cv_scores_mean']:.3f} "
          f"(±{results['optimization_results']['cv_scores_std']:.3f})")
    print(f"\nNumber of selected features: {len(results['optimization_results']['selected_features'])}")

    print("\nOverall Performance:")
    print("-------------------")
    print(pd.DataFrame(results['overall_report']).T)

    print("\nTop 10 Most Important SNPs:")
    print("-------------------------")
    print(results['feature_importance'].head(10))

    print("\nConfusion Matrix:")
    print("----------------")
    print(results['confusion_matrix'])

def print_optimization_results(results):
    """Print comprehensive optimization and evaluation results"""
    print("Model Type:", results['optimization_results']['model_type'])
    print("\nModel Optimization Results:")
    print("--------------------------")
    print(f"Best Parameters: {results['optimization_results']['best_params']}")
    print(f"Cross-validation Score: {results['optimization_results']['cv_scores_mean']:.3f} "
          f"(±{results['optimization_results']['cv_scores_std']:.3f})")

    print("\nOverall Performance:")
    print("-------------------")
    print(pd.DataFrame(results['overall_report']).T)

    print("\nTop 10 Most Important SNPs:")
    print("-------------------------")
    print(results['feature_importance'].head(10))

    print("\nConfusion Matrix:")
    print("----------------")
    print(results['confusion_matrix'])


In [ ]:
# test_data = pd.read_csv("./test_data.csv", index_col=0)

In [ ]:
# df_vampir_missing = simulate_test_missingness(df_vampir.T, test_data.T)

In [ ]:
taxon_labels = df_samples.taxon.to_numpy()
results = analyze_taxon_prediction(df_vampir, taxon_labels, model_type='xgboost', test_size=0.2)
print_optimization_results(results)

In [ ]:
a = results

import pickle

with open('./xg_taxon.pickle', 'wb') as handle:
    pickle.dump(a, handle, protocol=pickle.HIGHEST_PROTOCOL)